## vLLM Inferenz Server 

Beispiel für einen **vLLM-Client**, der mit einem **vLLM Inference Server auf einem separaten Rechner** kommuniziert.

Auf diesem Server läuft **vLLM** als zentral betriebener Inference-Service und stellt das geladene Modell über eine **OpenAI-kompatible API** für mehrere Clients bereit. Im Unterschied zu Ollama unterstützt **vLLM pro Container in der Regel nur ein einzelnes LLM**. Sollen mehrere Modelle parallel betrieben werden, werden dafür üblicherweise mehrere Container beziehungsweise mehrere getrennte Server-Instanzen verwendet.

Der folgende Code zeigt, wie aus einem **Python-Jupyter-Notebook** über diese OpenAI-kompatible API auf einen betriebenen vLLM-Service zugegriffen wird.

Die Verbindung erfolgt über den API-Endpoint des Servers. Der verwendete API-Key dient dabei lediglich als Platzhalter.

Nach dem setzen der Umgebungsvariablen Deployen wir den Inferenz Server mittels `helm`

In [ ]:
%%bash
git clone https://github.com/vllm-project/vllm.git

In [ ]:
%%bash
source ~/data/env.py
cat <<EOF | tee ~/data/env.py
OPENAI_API_KEY="$OPENAI_API_KEY"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="Qwen/Qwen3.5-4B"
AI_NAME="qwen"
AI_IP="${AI_IP}"
EOF

In [ ]:
%%bash
source ~/data/env.py
helm ${AI_KUBECONFIG} upgrade --install ${AI_NAME}-vllm \
  vllm/examples/online_serving/chart-helm \
  -n vllm --create-namespace \
  --set image.repository=vllm/vllm-openai \
  --set image.tag=v0.18.0 \
  --set-json 'gpuModels=["NVIDIA-GB10-SHARED"]' \
  --set-json 'image.command=["vllm","serve","Qwen/Qwen3.5-4B","--host","0.0.0.0","--port","8000","--dtype","bfloat16","--gpu-memory-utilization","0.40","--max-model-len","8192"]' \
  --set replicaCount=1 \
  --set resources.requests.cpu=8 \
  --set resources.requests.memory=16Gi \
  --set resources.requests.nvidia\.com/gpu=1 \
  --set resources.limits.cpu=8 \
  --set resources.limits.memory=16Gi \
  --set resources.limits.nvidia\.com/gpu=1 \
  --set extraInit.modelDownload.enabled=false \
  --set extraInit.pvcStorage=20Gi \
  --set readinessProbe.initialDelaySeconds=240 \
  --set livenessProbe.initialDelaySeconds=480


Bevor wir den einfachen Test starten können, muss der Inferenz Server bereit sein:

In [ ]:
%%bash
source ~/data/env.py
kubectl ${AI_KUBECONFIG} -n vllm get pods,services
# kubectl ${AI_KUBECONFIG} -n vllm describe deployment/${AI_NAME}-vllm-deployment-vllm
kubectl ${AI_KUBECONFIG} -n vllm logs deployment/${AI_NAME}-vllm-deployment-vllm --tail=10 || true
# kubectl ${AI_KUBECONFIG} -n vllm logs deployment/${AI_NAME}-vllm-deployment-vllm -f

### Testen

Vorher muss base_url gesetzt werden.

In [ ]:
%%bash
source ~/data/env.py
kubectl ${AI_KUBECONFIG} patch svc "${AI_NAME}-vllm-service" -n vllm -p '{"spec":{"type":"NodePort"}}'
echo "AI_BASE_URL=\"http://${AI_IP}:$(kubectl $AI_KUBECONFIG get -n vllm service ${AI_NAME}-vllm-service -o jsonpath='{.spec.ports[0].nodePort}')/v1\"" | tee -a ~/data/env.py

In [ ]:
%run ~/data/env.py
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY, base_url=AI_BASE_URL)

response = client.responses.create(
    model=AI_MODEL,
    input="Wer war John F. Kennedy?"
)

print(response.output_text)

---

## Weitere LLMs 

Deployt weitere LLMs und versucht obigen Prompt, Analysiert die Ausgabe.

Mögliche LLMs sind:
* **HuggingFaceTB/SmolLM2-135M-Instruct** - sehr kleines Modell neigt zu Halluzinationen
* **Qwen2.5-Coder-0.5B** - grösser, gut zum Entwicklern von Software
* **TinyLlama-1.1B-Chat-v1.0** - ebenfalls grösser, für allgemeine Textabfragen
* **Qwen2.5-7B-Instruct** - für allgemeine Textabfragen bei Helm muss zusätzlich `--set model.memFraction=0.40` mitgegeben werden.
* **swiss-ai/Apertus-8B-Instruct-2509** - Schweizer LLM, komplett OpenSource. Bei Helm muss zusätzlich `--set model.memFraction=0.40` mitgegeben werden.

**Hinweis**: neben dem Modell muss ich der Modellname angepasst werden.

---
### Weitere Beispiele

Wechselt in das Verzeichnis `03-openai` und spielt die Übungen durch. 

Welche Funktionieren und welche nicht?

---
### Aufräumen

In [ ]:
%%bash
source ~/data/env.py
helm ${AI_KUBECONFIG} -n vllm uninstall ${AI_NAME}-vllm || true
kubectl delete ns vllm || true